# E3-G — Comparaison statistique appariée à information multidevices identique (C2)

**Sources** : E3-E2 (Ridge/RF/XGB 105 entrées), E3-F2A (GCN 200 époques), E3-F2B (LSTM/TCN 200 époques).

**Question** : les différences de RMSE sur les mêmes 13 643 cibles par horizon résistent-elles à une rééchantillonnage temporel ?

**Méthode préspécifiée** :
- Unité de rééchantillonnage : **jour UTC entier**, regroupant toutes les observations et tous les dispositifs du jour ; bootstrap circulaire par blocs de **2 jours consécutifs**, 2000 réplications.
- À chaque réplication, les mêmes jours sont sélectionnés pour les deux modèles d'une paire ; les **trois graines** des modèles profonds sont conservées, et l'on calcule la **moyenne des RMSE par graine**. Les modèles déterministes sont évalués une fois.
- Contraste : `RMSE(A) − RMSE(B)` ; **négatif signifie A meilleur**.
- Intervalle percentile bilatéral à 95 %. Ce bootstrap mesure surtout l'incertitude liée à la période de test ; les trois graines ne constituent **pas** un échantillon suffisant pour caractériser complètement la variabilité d'initialisation.
- Analyses principales définies à l'avance : `GCN opérationnel − GCN shuffled`, `TCN − GCN opérationnel`, `GCN natif − persistance`, `Ridge − GCN natif`, `Ridge − persistance`. Analyses exploratoires supplémentaires possibles, signalées séparément.
- **Aucun test d'hypothèse ou choix de modèle n'est fondé sur le test** ; l'IC sert à décrire la robustesse des contrastes, pas à effectuer une nouvelle sélection d'hyperparamètres.
- Si l'IC traverse zéro, ne pas affirmer que A est supérieur à B. Les horizons et contrastes multiples ne sont pas corrigés pour multiplicité.

Les valeurs brutes, prédictions et résultats historiques ne sont pas modifiés.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json, platform
ROOT=Path.cwd().resolve()
OUTPUTS=ROOT/'outputs'
OUT=OUTPUTS/'E3_G_PAIRED_BLOCK_BOOTSTRAP'
SOURCES={
 'tabular':OUTPUTS/'E3_E2_multidevice_tabular_105'/'03_common_test_predictions.csv',
 'gcn':OUTPUTS/'E3_F2A_GCN_200epochs'/'03_common_test_predictions_all_seeds.csv',
 'sequence':OUTPUTS/'E3_F2B_LSTM_TCN_200epochs'/'03_common_test_predictions_all_seeds.csv'
}
B=2000
BLOCK_DAYS=2
RNG_SEED=20261104
SEEDS=(42,123,2026)
HORIZONS=(15,30,60,120)
PAIRS=[
 ('G_operational','G_shuffled'),
 ('TCN','G_operational'),
 ('G_native','Persistence'),
 ('Ridge','G_native'),
 ('Ridge','Persistence')
]
assert all(p.is_file() for p in SOURCES.values()),'Fichiers introuvables : '+str({k:str(v) for k,v in SOURCES.items() if not v.is_file()})
OUT.mkdir(parents=True,exist_ok=True)


In [ ]:
def ingest(path,kind):
    df=pd.read_csv(path)
    if kind=='gcn':df=df.rename(columns={'topology':'model'})
    if kind=='tabular':df['seed']=-1
    required={'uuid','timestamp','horizon_min','model','seed','y_true','y_pred','persistence_pred'}
    assert required.issubset(df.columns),(kind,required-set(df.columns))
    df['timestamp']=pd.to_datetime(df.timestamp,utc=True)
    df['uuid']=df.uuid.astype(str)
    df['model']=df.model.astype(str)
    df['seed']=df.seed.astype(int)
    assert df[['y_true','y_pred','persistence_pred']].notna().all().all()
    assert np.isfinite(df[['y_true','y_pred','persistence_pred']].to_numpy()).all()
    assert not df.duplicated(['horizon_min','model','seed','uuid','timestamp']).any()
    return df[['uuid','timestamp','horizon_min','model','seed','y_true','y_pred','persistence_pred']]

data=pd.concat([ingest(path,k) for k,path in SOURCES.items()],ignore_index=True)
assert not data.duplicated(['horizon_min','model','seed','uuid','timestamp']).any()
print(data.groupby(['model','horizon_min']).agg(seeds=('seed','nunique'),rows=('y_true','size')).to_string())


In [ ]:
# Alignement strict des cibles et de la persistance, pour chaque horizon et chaque modèle/graine.
audit=[]
reference={}
for h in HORIZONS:
    sub=data[data.horizon_min.eq(h)]
    ref=(sub[(sub.model=='Ridge') & (sub.seed==-1)]
         .sort_values(['uuid','timestamp']).reset_index(drop=True))
    assert len(ref)==13643,f'Ridge reference missing for {h}'
    key=ref[['uuid','timestamp']]
    for (model,seed),g in sub.groupby(['model','seed']):
        g=g.sort_values(['uuid','timestamp']).reset_index(drop=True)
        assert len(g)==len(ref),(h,model,seed,len(g))
        assert g[['uuid','timestamp']].equals(key),(h,model,seed,'key mismatch')
        assert np.allclose(g.y_true,ref.y_true,atol=1e-5,rtol=0),(h,model,seed,'target mismatch')
        assert np.allclose(g.persistence_pred,ref.persistence_pred,atol=1e-5,rtol=0),(h,model,seed,'persistence mismatch')
        audit.append({'horizon_min':h,'model':model,'seed':seed,'n':len(g),
                      'keys_aligned':True,'targets_aligned':True,'persistence_aligned':True})
    reference[h]=ref
assert set(data.model.unique())=={'Ridge','RandomForest','XGBoost','LSTM','TCN','G_native','G_operational','G_shuffled'}
assert len(audit)==4*(3+5*3)
pd.DataFrame(audit).to_csv(OUT/'01_alignment_audit.csv',index=False)
print('Alignment checks passed:',len(audit))


## Bootstrap temporel

La RMSE est calculée à partir de la somme des erreurs quadratiques et du nombre d'observations des jours rééchantillonnés. Les jours consécutifs sont sélectionnés **avec la même multiplicité** pour tous les modèles d'un contraste, préservant ainsi l'appariement. Les observations des sept dispositifs d'un même jour restent groupées.


In [ ]:
def rmse(v):return float(np.sqrt(np.mean(np.square(v))))
rng=np.random.default_rng(RNG_SEED)
metric_rows=[];contrast_rows=[];daily_rows=[]
for h in HORIZONS:
    ref=reference[h]
    # Sorted by uuid/timestamp (same ordering in all sources); create per-day summaries.
    dates=ref.timestamp.dt.floor('D')
    unique_days=pd.DatetimeIndex(sorted(dates.unique()))
    D=len(unique_days)
    assert D>=4,f'Too few days for temporal bootstrap h={h}: {D}'
    day_codes=unique_days.get_indexer(dates)
    day_n=np.bincount(day_codes,minlength=D).astype(float)
    assert day_n.sum()==13643
    models={}
    for model in sorted(data.model.unique()):
        subset=data[(data.horizon_min==h)&(data.model==model)]
        seeds=sorted(subset.seed.unique())
        arr=[]
        for seed in seeds:
            g=subset[subset.seed==seed].sort_values(['uuid','timestamp'])
            assert len(g)==len(ref)
            sq=(g.y_pred.to_numpy()-ref.y_true.to_numpy())**2
            arr.append(np.bincount(day_codes,weights=sq,minlength=D))
        models[model]=np.stack(arr) # [n_seeds, n_days]
        per_seed=np.sqrt(models[model].sum(axis=1)/day_n.sum())
        metric_rows.append({'horizon_min':h,'model':model,'n_days':D,'n_nodes':int(day_n.sum()),
                            'n_seeds':len(seeds),'RMSE_mean':float(per_seed.mean()),
                            'RMSE_SD':float(per_seed.std(ddof=1)) if len(seeds)>1 else np.nan})
    persistence_sq=(ref.persistence_pred.to_numpy()-ref.y_true.to_numpy())**2
    models['Persistence']=np.bincount(day_codes,weights=persistence_sq,minlength=D)[None,:]
    metric_rows.append({'horizon_min':h,'model':'Persistence','n_days':D,'n_nodes':13643,
                        'n_seeds':1,'RMSE_mean':rmse(ref.persistence_pred.to_numpy()-ref.y_true.to_numpy()),
                        'RMSE_SD':np.nan})
    # Circular moving-block bootstrap: each replicate comprises ceil(D/L) sampled blocks, truncated to D days.
    nblocks=int(np.ceil(D/BLOCK_DAYS))
    starts=rng.integers(0,D,size=(B,nblocks))
    offsets=np.arange(BLOCK_DAYS)
    picks=((starts[:,:,None]+offsets)%D).reshape(B,-1)[:,:D]
    counts=np.zeros((B,D),dtype=np.int32)
    np.add.at(counts,(np.repeat(np.arange(B),D),picks.reshape(-1)),1)
    boot_n=counts@day_n
    assert np.all(boot_n>0)
    boot_rmse={}
    for model,sqday in models.items():
        # shape: seeds × replicates, then average RMSE over seeds (not average predictions).
        values=np.sqrt((sqday@counts.T)/boot_n[None,:])
        boot_rmse[model]=values.mean(axis=0)
    for a,b in PAIRS:
        observed=float(np.mean(np.sqrt(models[a].sum(axis=1)/day_n.sum()))-
                       np.mean(np.sqrt(models[b].sum(axis=1)/day_n.sum())))
        dist=boot_rmse[a]-boot_rmse[b]
        low,high=np.quantile(dist,[0.025,0.975])
        contrast_rows.append({'horizon_min':h,'A':a,'B':b,'contrast':'RMSE(A)-RMSE(B)',
                              'delta_RMSE':observed,'CI95_low':float(low),'CI95_high':float(high),
                              'fraction_bootstrap_A_better':float(np.mean(dist<0)),
                              'CI_excludes_zero':bool(low>0 or high<0),
                              'n_days':D,'block_days':BLOCK_DAYS,'bootstrap_replicates':B,
                              'interpretation':'A lower RMSE' if observed<0 else 'B lower RMSE'})
    print(f'h={h}, days={D}, contrasts={len(PAIRS)}')
metrics_df=pd.DataFrame(metric_rows)
contrasts_df=pd.DataFrame(contrast_rows)
metrics_df.to_csv(OUT/'02_model_metrics_common_test.csv',index=False)
contrasts_df.to_csv(OUT/'03_PAIRED_block_bootstrap_CI.csv',index=False)
print(contrasts_df[['horizon_min','A','B','delta_RMSE','CI95_low','CI95_high','CI_excludes_zero']].to_string(index=False))


In [ ]:
# Audit interne : valeurs bootstrap = mêmes RMSE moyennes que fichiers d'origine, à précision numérique près.
checks=[]
for h in HORIZONS:
    for model in ('Ridge','TCN','G_native','G_operational','G_shuffled'):
        rows=data[(data.horizon_min==h)&(data.model==model)]
        r=rows.groupby('seed').apply(lambda g:rmse(g.y_pred.to_numpy()-g.y_true.to_numpy()),include_groups=False).mean()
        got=metrics_df.query('horizon_min==@h and model==@model').RMSE_mean.iloc[0]
        assert abs(r-got)<1e-9,(h,model,r,got)
        checks.append({'horizon_min':h,'model':model,'RMSE_direct':r,'RMSE_from_daily_sums':got})
pd.DataFrame(checks).to_csv(OUT/'04_recomputation_checks.csv',index=False)
protocol={'experiment':'E3-G C2 paired day-block bootstrap','bootstrap_replicates':B,
          'block_days':BLOCK_DAYS,'rng_seed':RNG_SEED,'horizons':list(HORIZONS),
          'contrasts':PAIRS,'sources':{k:str(v) for k,v in SOURCES.items()},
          'unit':'UTC day; all 7 devices together',
          'seed_handling':'mean of per-seed RMSE, fixed three training seeds; deterministic models once',
          'CI':'percentile 95%, descriptive, no multiple-comparison adjustment',
          'limitations':['single chronological test period','only three seeds',
                         'bootstrap block length sensitivity not yet assessed',
                         'no inference on semantic-layer causal contribution'],
          'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__}
(OUT/'05_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print('Saved:',OUT)


## Interprétation et suite

1. **Contrôle obligatoire** : `01_alignment_audit.csv` et `04_recomputation_checks.csv` doivent être entièrement conformes.
2. **Résultat principal** : `03_PAIRED_block_bootstrap_CI.csv` : un contraste négatif favorise A ; ne pas interpréter un IC incluant zéro comme une preuve d'égalité.
3. **Analyse de sensibilité recommandée** : répéter le protocole avec des blocs de 1, 3 et 5 jours, sans modifier les modèles.
4. **Limites** : l'IC décrit la sensibilité au choix des jours de test, et non la généralisation à d'autres bâtiments ; il ne démontre pas l'utilité causale de la sémantique. Les observations du même jour et des sept dispositifs sont groupées, mais la dépendance entre blocs éloignés n'est pas complètement éliminée.
5. **Fichiers à transmettre** : `01_alignment_audit.csv`, `02_model_metrics_common_test.csv`, `03_PAIRED_block_bootstrap_CI.csv`, `04_recomputation_checks.csv`, `05_protocol.json`.
